<a href="https://colab.research.google.com/github/Tasee-P/Final502/blob/main/ViZ502_final_Phacha.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import pandas as pd

# the path for three files
path_cpi = '/content/drive/MyDrive/ti-corruption-perception-index.csv'
path_gni = '/content/drive/MyDrive/API_NY.GNP.PCAP.CD_DS2_en_csv_v2_230.csv'
path_pop = '/content/drive/MyDrive/POP_DPND.csv'

# load each file into df
# Note:  skip the first 4 rows for the World Bank files because they contain extra header text
df_cpi = pd.read_csv(path_cpi)
df_gni = pd.read_csv(path_gni, skiprows=4)
df_pop = pd.read_csv(path_pop, skiprows=4)

# check if they loaded correctly
print("CPI Data loaded successfully!")
print("GNI Data loaded successfully!")
print("OADR Data loaded successfully!")
df_cpi.head(2)
df_gni.head(2)
df_pop.head(2)

CPI Data loaded successfully!
GNI Data loaded successfully!
OADR Data loaded successfully!


,Country Name,Country Code,Indicator Name,Indicator Code,1960,1961,1962,1963,1964,1965,...,2016,2017,2018,2019,2020,2021,2022,2023,2024,Unnamed: 69
0,Aruba,ABW,"Age dependency ratio, old (% of working-age po...",SP.POP.DPND.OL,5.229128,5.222317,5.256221,5.302170,5.382927,5.500667,...,17.655933,18.582775,19.582866,20.647987,21.701218,22.590708,23.519229,24.635873,25.804463,NaN
1,Africa Eastern and Southern,AFE,"Age dependency ratio, old (% of working-age po...",SP.POP.DPND.OL,5.631545,5.607333,5.585546,5.570575,5.562311,5.562341,...,5.591522,5.623820,5.668747,5.721924,5.759325,5.777143,5.805446,5.853275,5.909405,NaN


In [41]:
!pip install -U kaleido

In [3]:
import pandas as pd
import numpy as np
import plotly.express as px

# Define Paths
path_cpi = '/content/drive/MyDrive/ti-corruption-perception-index.csv'
path_gni = '/content/drive/MyDrive/API_NY.GNP.PCAP.CD_DS2_en_csv_v2_230.csv'
path_pop = '/content/drive/MyDrive/POP_DPND.csv'

# load each file into df58
df_cpi_58 = pd.read_csv(path_cpi)
df_gni_58 = pd.read_csv(path_gni, skiprows=4)
df_pop_58 = pd.read_csv(path_pop, skiprows=4)

# clean and prepare Ddata
#target_year_wb = '2023'
#target_year_cpi = 2023
target_year_wb = '2024'
target_year_cpi = 2024


cpi_clean_58 = df_cpi_58[df_cpi_58['Year'] == target_year_cpi][['Entity', 'Code', 'Corruption Perceptions Index']].copy()
cpi_clean_58.rename(columns={'Corruption Perceptions Index': 'CPI', 'Entity': 'Country Name'}, inplace=True)

pop_clean_58 = df_pop_58[['Country Code', target_year_wb]].copy()
pop_clean_58.rename(columns={target_year_wb: 'Dependency_Ratio'}, inplace=True)

gni_clean_58 = df_gni_58[['Country Code', target_year_wb]].copy()
gni_clean_58.rename(columns={target_year_wb: 'GNI_Per_Capita'}, inplace=True)

df_merged_58 = cpi_clean_58.merge(pop_clean_58, left_on='Code', right_on='Country Code', how='inner')
df_merged_58 = df_merged_58.merge(gni_clean_58, on='Country Code', how='inner')
df_merged_58.dropna(subset=['CPI', 'Dependency_Ratio', 'GNI_Per_Capita'], inplace=True)

# assign income levels
conditions_income = [
    #df_merged_58['GNI_Per_Capita'] >= 13846,
    #(df_merged_58['GNI_Per_Capita'] >= 1136) & (df_merged_58['GNI_Per_Capita'] < 13846),
    #df_merged_58['GNI_Per_Capita'] < 1136
    df_merged_58['GNI_Per_Capita'] >= 14005,
    (df_merged_58['GNI_Per_Capita'] >= 1145) & (df_merged_58['GNI_Per_Capita'] < 14005),
    df_merged_58['GNI_Per_Capita'] < 1145


]
choices_income = ['High', 'Mid', 'Low']
df_merged_58['Income_Level'] = np.select(conditions_income, choices_income, default='Other')

# Use age group logic
conditions_category = [
    df_merged_58['Dependency_Ratio'] < 15,
    (df_merged_58['Dependency_Ratio'] >= 15) & (df_merged_58['Dependency_Ratio'] < 20),
    (df_merged_58['Dependency_Ratio'] >= 30) & (df_merged_58['CPI'] >= 50) & (df_merged_58['Income_Level'] == 'High'),
    (df_merged_58['Dependency_Ratio'] >= 20) & (df_merged_58['Dependency_Ratio'] < 30) & (df_merged_58['CPI'] >= 50) & (df_merged_58['Income_Level'] == 'High'),
    (df_merged_58['Dependency_Ratio'] >= 20) & (df_merged_58['CPI'] >= 50) & (df_merged_58['Income_Level'].isin(['Mid', 'Low'])),
    (df_merged_58['Dependency_Ratio'] >= 20) & (df_merged_58['CPI'] < 50) & (df_merged_58['Income_Level'] == 'High'),
    (df_merged_58['Dependency_Ratio'] >= 20) & (df_merged_58['CPI'] < 50) & (df_merged_58['Income_Level'] == 'Mid'),
    (df_merged_58['Dependency_Ratio'] >= 20) & (df_merged_58['CPI'] < 50) & (df_merged_58['Income_Level'] == 'Low')
]

choices_category = [
    "The Young World<br>(< 15% OADR)",
    "Approaching Aging<br>(15-20% OADR)",
    "Severe Aging<br>Clean, Rich",
    "Aging<br>Clean, Rich",
    "Aging/Severe<br>Clean, Mid/Low",
    "Aging/Severe<br>Corrupt, Rich",
    "Aging/Severe<br>Corrupt, Mid",
    "Aging/Severe<br>Corrupt, Low"
]

df_merged_58['Category'] = np.select(conditions_category, choices_category, default='Uncategorized')
df_merged_58 = df_merged_58[df_merged_58['Category'] != 'Uncategorized']

# after run this code, i found group some group i design does not exist, so fix and  only keep categories that actually have data so empty ones are dropped
actual_categories = df_merged_58['Category'].unique()
master_order = [
    "Severe Aging<br>Clean, Rich", "Aging<br>Clean, Rich", "Aging/Severe<br>Clean, Mid/Low",
    "Aging/Severe<br>Corrupt, Rich", "Aging/Severe<br>Corrupt, Mid", "Aging/Severe<br>Corrupt, Low",
    "Approaching Aging<br>(15-20% OADR)", "The Young World<br>(< 15% OADR)"
]
category_order = [cat for cat in master_order if cat in actual_categories]

# interactive plotly boxplot
fig_58 = px.box(
    df_merged_58,
    x='Category',
    y='Dependency_Ratio',
    color='Category',
    points='all',
    hover_data=['Country Name', 'CPI', 'GNI_Per_Capita', 'Income_Level'],
    category_orders={"Category": category_order},

    color_discrete_map={
        "Severe Aging<br>Clean, Rich": "#1E3A8A",
        "Aging<br>Clean, Rich": "#38BDF8",
        "Aging/Severe<br>Clean, Mid/Low": "#059669",
        "Aging/Severe<br>Corrupt, Rich": "#D97706",
        "Aging/Severe<br>Corrupt, Mid": "#EA580C",
        "Aging/Severe<br>Corrupt, Low": "#DC2626",
        "Approaching Aging<br>(15-20% OADR)": "#8B5CF6",
        "The Young World<br>(< 15% OADR)": "#9CA3AF"
    }
)

fig_58.update_traces(width=0.45, marker=dict(size=6, opacity=0.85))

# polish visual formatting (biggertext,  biggerplot)
fig_58.update_layout(
    plot_bgcolor='#FAFAFA', #  subtle off-white background
    paper_bgcolor='white',
    yaxis=dict(
        title=dict(text='Old Age Dependency Ratio (%)', font=dict(color='black', size=18, family='Arial')),
        gridcolor='lightgray',
        gridwidth=1,
        griddash='dot',
        tickfont=dict(color='gray', size=16, family='Arial')
    ),
    xaxis=dict(
        title=None,
        tickangle=0,
        tickfont=dict(color='black', size=16, family='Arial')
    ),
    showlegend=False,
    width=1500,
    height=900,
    title=dict(
        text="Demographic Destiny: Wealth, Governance, and the Global Aging Curve<br><sup>High-income, transparent nations lead the aging trend, while corrupt and developing economies face a looming demographic trap.</sup>",
        font=dict(size=28, color='black', family='Arial'),
        x=0.05
    ),
    margin=dict(b=120, t=120)
)

# add annotation
fig_58.add_annotation(
    text="Source: World Bank Global & Transparency International, 2024.",
    xref="paper", yref="paper",
    x=1.0, y=-0.15,
    showarrow=False,
    font=dict(size=13, color="gray"),
    xanchor="right", yanchor="top"
)

fig_58.show()


# save as an interactive HTML file
fig_58.write_html('/content/drive/MyDrive/Demographic_Chart_Final.html')


# save as a high-resolution PNG image (Scale=2 makes it ultra crisp)
#fig_58.write_image('/content/drive/MyDrive/Demographic_Chart_Final.png', scale=2)


In [46]:
# check specific data and category from the boxplot.
check = df_merged_58[df_merged_58['Country Name'] == 'Romania']

display(check[['Country Name', 'Dependency_Ratio', 'CPI', 'GNI_Per_Capita', 'Income_Level', 'Category']])

,Country Name,Dependency_Ratio,CPI,GNI_Per_Capita,Income_Level,Category
131,Romania,31.073085,46,17600.0,High,"Aging/Severe<br>Corrupt, Rich"


In [16]:
import pandas as pd
import plotly.express as px

# list of the  selected countries


selected_countries = [
    'USA',
    #'CAN',
    'GBR',
    'DEU',
    'CHN',
    'JPN',
    #'AUS',
    #'BRA',
    'MEX',
    'ZAF',
    'NGA',
    'IND',
    'THA',
    'GTM',
    'ARE',
    'KAZ',
    'TUR',
    'UZB',
    'KOR',
    'CHL',
    'ARG',
    'RUS',
    'IRN',
    'MYS',
    'BGD',
    'ITA',
    'ESP',
    'GRC',
    'UKR',
    'MDA',
    'MMR',
    'SGP',
    #'TUN',
    'NZL',
    'BGR',
    'GEO',
    'HRV',
    'EST',
    'FRA',
    'DNK',
    'SRB',
    'FIN',
    'PRT',
    'NOR',
    #'BIH'

]

#need new var nam
df_cpi_c1 = df_cpi.copy()
df_gni_c1 = df_gni.copy()
df_pop_c1 = df_pop.copy()

def melt_wb_c1(df, value_name):
    years = [str(y) for y in range(2012, 2025) if str(y) in df.columns]
    df_melt = df.melt(id_vars=['Country Code'], value_vars=years, var_name='Year', value_name=value_name)
    df_melt['Year'] = df_melt['Year'].astype(int)
    return df_melt

df_gni_melt_c1 = melt_wb_c1(df_gni_c1, 'GNI_per_capita')
df_pop_melt_c1 = melt_wb_c1(df_pop_c1, 'Age_Dependency')

# clean CPI data
df_cpi_c1 = df_cpi_c1.rename(columns={'Code': 'Country Code', 'Corruption Perceptions Index': 'CPI', 'Entity': 'Country'})
df_cpi_c1 = df_cpi_c1.dropna(subset=['Country Code'])
df_cpi_c1 = df_cpi_c1[(df_cpi_c1['Year'] >= 2012)]

# merge Data into df_chart1
df_chart1 = df_cpi_c1[['Country', 'Country Code', 'Year', 'CPI']].merge(df_gni_melt_c1, on=['Country Code', 'Year'], how='inner')
df_chart1 = df_chart1.merge(df_pop_melt_c1, on=['Country Code', 'Year'], how='inner')
df_chart1 = df_chart1.dropna(subset=['CPI', 'GNI_per_capita', 'Age_Dependency'])

# filter to countries selected
df_chart1 = df_chart1[df_chart1['Country Code'].isin(selected_countries)]

# ensure data is sorted for smooth animation frames
df_chart1 = df_chart1.sort_values(by=['Year', 'Country'])

# costom color
custom_colors = {
    #'USA': 'blue', 'CAN': 'lightblue', 'GBR': 'red', 'DEU': 'black',
    #'CHN': 'darkred', 'JPN': 'pink', 'AUS': 'orange', 'BRA': 'green',
    #'MEX': 'darkgreen', 'ZAF': 'gold', 'NGA': 'darkorange', 'IND': 'purple'

    'USA': '#9CA3AF',  ##1D3557 Deep Navy Blue to grey
    #'CAN': '#00B4D8',  # Bright Cyan
    'GBR': '#E63946',  # Crimson Red
    'DEU': '#C2BEB3',  # Charcoal Black#2A2A2A',
    'CHN': '#F4A261',  # Muted Orange
    'JPN': '#FF006E',  # Vibrant Magenta
    #'AUS': '#E9C46A',  # Mustard Yellow
    #'BRA': '#2A9D8F',  # Deep Teal
    'MEX': '#A7C957',  # Lime Green
    'ZAF': '#6D597A',  # Muted Purple
    'NGA': '#7F4F24',  # Earthy Brown
    'IND': '#3A0CA3',  # Deep Indigo
    'THA': '#00A8E8',  # Mint Green to #06D6A0 Azue Blue
    'GTM': '#118AB2',  # Ocean Blue
    'ARE': '#B5179E',  # Deep Fuchsia
    'KAZ': '#F15BB5',  # Bright Pink
    'TUR': '#E63946',
    'UZB': '#00F5D4',   # Aquamarine
    'KOR': '#4361EE',  # Electric Royal Blue
    'CHL': '#FFB703',  # Vivid Amber
    'ARG': '#FF6B6B',  # Warm Coral
    'RUS': '#C1121F',  # Red
    "IRN": '#00A676',   # Emerald Green
    'MYS': '#9D4EDD',
    'BGD': '#118C4F',
    'ITA': '#FFC300',
    'ESP': '#E07A5F',
    'GRC': '#E5989B',
    'UKR': '#FCA311',
    'MDA': '#E5989B',
    'MMR': '#2A9D8F',
    'SGP': '#FCA311',
    #'TUN': '#D62828',
    'NZL': '#C2BEB3',
    'BGR': '#9C27B0',  # Deep Purple (Bulgaria)
    'GEO': '#009688',  # Vibrant Teal (Georgia)
    'HRV': '#FF9800',
    'EST': '#F4A261',  # Warm Sand/Orange (Estonia)
    'FRA': '#1D3557',  # Deep Navy Blue (France)
    'DNK': '#2A9D8F',
    'SRB': '#D81B60',
    'FIN': '#00BCD4',  # Cyan/Ice Blue (Finland)
    'PRT': '#4CAF50',  # Vibrant Green (Portugal)
    'NOR': '#8BC34A',
    #'BIH': '#008080'

}

# create interactive plot
fig1 = px.scatter(
    df_chart1,
    x="Age_Dependency",
    y="CPI",
    animation_frame="Year",
    animation_group="Country",
    size="GNI_per_capita",
    color="Country Code",
    color_discrete_map=custom_colors,
    text="Country",
    hover_name="Country",
    hover_data={
        "Age_Dependency": ':.2f',
        "GNI_per_capita": True,
        "Country Code": False,
        "Country": False
    },
    size_max=65,
    range_y=[0, 115],
    range_x=[-2, 60],
    title="<b>Navigating the Tripartite Model: Wealth, Aging,and Corruption.</b><br><sup>How national wealth, demographic shifts, and corruption intersect across global economies.</sup>",
    labels={
        "CPI": "Corruption Score (Points, Higher = More Transparent)",
        "Age_Dependency": "Old-Age Dependency (%)",
        "GNI_per_capita": "GNI per Capita (US$)"
    }
)

#fig1.update_traces(textposition='top center', cliponaxis=False, textfont_size=16)
fig1.update_traces(
    textposition='top center',
    cliponaxis=False,
    textfont_size=16,
    marker=dict(line=dict(width=1.5, color='white')) # <--- ADD THIS: Creates a sharp white border around every bubble
)

# fixed Layout & slider adjustments
# this part is a bit problems need the right  layout size  to make it looks nice
fig1.update_layout(
    height=900,
    #width=1500,
    font=dict(size=16.5),
    title_font=dict(size=26),        # scales up the main title and the subtitle
    plot_bgcolor='#F0F4F8',          # blue-gray for the chart area
    paper_bgcolor='#F0F4F8',

    # fix the ticks so they don't overlap at the corner
    xaxis=dict(showgrid=True, gridwidth=1, gridcolor='LightGray', tickvals=[0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100]),
    yaxis=dict(showgrid=True, gridwidth=1, gridcolor='LightGray', tickvals=[20, 40, 60, 80, 100]), # Removed '0' here!
    margin=dict(t=180, b=160),
    updatemenus=[dict(y=-0.12)],
    sliders=[dict(y=-0.12, font=dict(size=18))],
    annotations=[
        dict(
            text="Source: Transparency International & World Bank, 2024 | Note: Bubble size = GNI per Capita",
            xref="paper", yref="paper",
            x=1.0,
            y=-0.22,
            showarrow=False,
            font=dict(size=14.5, color="gray"),
            xanchor="right"
        )
    ]
)

#fig1.write_html("interactive_bubble_chart_1.html")
fig1.write_html('/content/drive/MyDrive/Interactive_bubble_oadr_cpi_gni.html')

#picture
#fig1.write_html('/content/drive/MyDrive/Interactive_bubble_oadr_cpi_gni.png', scale=2)

fig1.show()




In [23]:
# I remember Taiwan show in CPI, but can not find in WB , Let check
print("World Bank Name:", df_pop[df_pop['Country Code'] == 'TWN']['Country Name'].unique())

# check CPI name
print("CPI Name:", df_cpi[df_cpi['Code'] == 'TWN']['Entity'].unique())

World Bank Name: []
CPI Name: ['Taiwan']


In [49]:
#I can not find Taiwan, so i search for 'Taiwan, does it exist anywhere in the the data
print("Searching World Bank Pop Data:")
print(df_pop[df_pop['Country Name'].str.contains("Taiwan", na=False)]['Country Name'].unique())

print("\nSearching World Bank GNI Data:")
print(df_gni[df_gni['Country Name'].str.contains("Taiwan", na=False)]['Country Name'].unique())

Searching World Bank Pop Data:
[]

Searching World Bank GNI Data:
[]


In [ ]:
# grab all the unique country names and sort them alphabetically
all_countries = sorted(df_pop8['Country Name'].dropna().unique())

# print the list to see every available name
for country in all_countries:
    print(country)

Afghanistan
Africa Eastern and Southern
Africa Western and Central
Albania
Algeria
American Samoa
Andorra
Angola
Antigua and Barbuda
Arab World
Argentina
Armenia
Aruba
Australia
Austria
Azerbaijan
Bahamas, The
Bahrain
Bangladesh
Barbados
Belarus
Belgium
Belize
Benin
Bermuda
Bhutan
Bolivia
Bosnia and Herzegovina
Botswana
Brazil
British Virgin Islands
Brunei Darussalam
Bulgaria
Burkina Faso
Burundi
Cabo Verde
Cambodia
Cameroon
Canada
Caribbean small states
Cayman Islands
Central African Republic
Central Europe and the Baltics
Chad
Channel Islands
Chile
China
Colombia
Comoros
Congo, Dem. Rep.
Congo, Rep.
Costa Rica
Cote d'Ivoire
Croatia
Cuba
Curacao
Cyprus
Czechia
Denmark
Djibouti
Dominica
Dominican Republic
Early-demographic dividend
East Asia & Pacific
East Asia & Pacific (IDA & IBRD countries)
East Asia & Pacific (excluding high income)
Ecuador
Egypt, Arab Rep.
El Salvador
Equatorial Guinea
Eritrea
Estonia
Eswatini
Ethiopia
Euro area
Europe & Central Asia
Europe & Central Asia (IDA & I

In [3]:
import pandas as pd
import plotly.express as px

# prepare all three datasets for the matching years
years = [str(y) for y in range(2012, 2025)]

# clean CPI
df_cpi_c = df_cpi.rename(columns={'Code': 'Country Code', 'Corruption Perceptions Index': 'CPI', 'Entity': 'Country'})
df_cpi_c = df_cpi_c.dropna(subset=['Country Code'])
df_cpi_c = df_cpi_c[(df_cpi_c['Year'] >= 2012) & (df_cpi_c['Year'] <= 2024)]

# clean GNI
years_gni = [y for y in years if y in df_gni.columns]
df_gni_melt = df_gni.melt(id_vars=['Country Code'], value_vars=years_gni, var_name='Year', value_name='GNI_per_capita')
df_gni_melt['Year'] = df_gni_melt['Year'].astype(int)

# clean OADR
years_pop = [y for y in years if y in df_pop.columns]
df_pop_melt = df_pop.melt(id_vars=['Country Code', 'Country Name'], value_vars=years_pop, var_name='Year', value_name='Age_Dependency')
df_pop_melt['Year'] = df_pop_melt['Year'].astype(int)
df_pop_melt = df_pop_melt.rename(columns={'Country Name': 'Country_Pop'}) # Rename to avoid merge conflicts

# Merge all three together
df_merged = pd.merge(df_cpi_c[['Country Code', 'Year', 'CPI', 'Country']], df_gni_melt, on=['Country Code', 'Year'], how='inner')
df_merged = pd.merge(df_merged, df_pop_melt[['Country Code', 'Year', 'Age_Dependency']], on=['Country Code', 'Year'], how='inner')
df_merged.dropna(subset=['CPI', 'GNI_per_capita', 'Age_Dependency'], inplace=True)

# countries selection
g7_codes = ['CAN', 'FRA', 'DEU', 'ITA', 'JPN', 'GBR', 'USA']
asean_codes = ['KHM', 'IDN', 'LAO', 'MYS', 'MMR', 'PHL', 'SGP', 'THA', 'VNM']
extra_codes = ['KOR', 'CHN', 'IND', 'BRA']
selective_codes = ['CHN', 'PHL', 'VNM', 'MMR', 'KAZ', 'THA','IDN','LKA', 'IND']
#TLS

#target_codes = list(set(asean_codes + extra_codes))
target_codes = list(set(selective_codes))
df_plot = df_merged[df_merged['Country Code'].isin(target_codes)].copy()

# normalize data, index to aoo in base year 2012

base_values = df_plot[df_plot['Year'] == 2012][['Country Code', 'CPI', 'GNI_per_capita', 'Age_Dependency']].rename(
    columns={'CPI': 'CPI_base', 'GNI_per_capita': 'GNI_base', 'Age_Dependency': 'Age_base'}
)
df_plot = df_plot.merge(base_values, on='Country Code', how='left')

# calculate the indexed values
df_plot['Corruption (CPI)'] = (df_plot['CPI'] / df_plot['CPI_base']) * 100
df_plot['Wealth (GNI)'] = (df_plot['GNI_per_capita'] / df_plot['GNI_base']) * 100
df_plot['Aging (Dependency)'] = (df_plot['Age_Dependency'] / df_plot['Age_base']) * 100

# create the data for all three lines
df_final = df_plot.melt(
    id_vars=['Country Code', 'Country', 'Year'],
    value_vars=['Corruption (CPI)', 'Wealth (GNI)', 'Aging (Dependency)'],
    var_name='Indicator',
    value_name='Index Value'
)

# sort alphabetically by country
df_final = df_final.sort_values(by=['Country', 'Year'])

ft_salmon = "#FFF1E5"
ft_text = "#33302E"
ft_muted = "#807973"

# create the small multiples line chart
fig = px.line(
    df_final,
    x="Year",
    y="Index Value",
    color="Indicator",
    facet_col="Country",
    facet_col_wrap=3,           # makes it 3x3
    #facet_col_spacing=0.08,     # wider gap between columns
    #facet_row_spacing=0.1,      # wider gap between rows for big titles

    facet_col_spacing=0.04,     # reduced from 0.08 to bring charts closer
    facet_row_spacing=0.07,     # reduced to tighten vertical gap


    height=1300,
   color_discrete_map={
        "Corruption (CPI)": "#4361EE",
        "Wealth (GNI)": "#2A9D8F",
        "Aging (Dependency)": "#9D4EDD"
    }

)

# text is too small, let fix text


main_title = "<b>Frontier vs. Fortune: The Asian Divergence</b>"
sub_title = "Relative growth trajectories of Wealth, Aging, and Corruption since 2012 (Base Year = 100)"

# work on title and legend
fig.update_layout(
    height=1600,
    paper_bgcolor="white",
    plot_bgcolor="#FFF1E5",
    margin=dict(t=320, b=150, l=100, r=40), # Increased top (t=320) and left (l=100) margins for bigger text

    #title=dict(
        # bigger on title and sub
        #text=f"<span style='font-size:64px; font-family:Arial Black;'>{main_title}</span><br><span style='font-size:36px; color:#807973;'>{sub_title}</span>",
       # x=0.03,
        #y=0.93,             # LOWERED from 0.98 to 0.93 so the top of the letters don't get cut off!
        #xanchor="left",
        #yanchor="top"
    title=dict(
        # I added an invisible 16px line break between the title and subtitle to space them out!
        text=f"<span style='font-size:64px; font-family:Arial Black;'>{main_title}</span><span style='font-size:10px;'><br><br></span><span style='font-size:36px; color:#807973;'>{sub_title}</span>",
        x=0.03,
        y=0.93,
        xanchor="left",
        yanchor="top"
    ),


    legend=dict(
        title="",
        font=dict(size=26),
        orientation="h",
        yanchor="bottom",
        y=1.06,
        xanchor="right",
        x=1.0
    ),
    template="plotly_white"
)

# increased size to match bigger axes
fig.for_each_annotation(lambda a: a.update(
    text=f"<b>{a.text.split('=')[-1]}</b>",
    font=dict(size=32, color=ft_text, family="Arial Black") # Increased from 24 to 32
))

#bigger andn remove year label
fig.update_xaxes(
    title_text="", # <-- This completely removes the word "Year"
    tickfont=dict(size=22, color=ft_text),
    showgrid=True, gridcolor='white',
    matches='x'
)

# bigger y axis
fig.update_yaxes(
    range=None,
    title_text="Index",
    title_font=dict(size=28, color=ft_text),
    tickfont=dict(size=22, color=ft_text),
    showgrid=True, gridcolor='white'
)

# baseline credit
fig.add_hline(y=100, line_dash="dot", line_width=2, line_color=ft_muted, opacity=0.8)

# bigger source
fig.add_annotation(
    text="Source: Transparency International & World Bank, 2024",
    xref="paper", yref="paper",
    x=1.0,
    y=-0.1,
    showarrow=False,
    font=dict(size=28, color="#807973"), # <-- Increased size from 18 to 28 so it's highly readable!
    xanchor="right",
    yanchor="top"
)
# Render / Save the interactive html


fig.write_html('/content/drive/MyDrive/Interactive_triple_line_chart.html')
# Save as a high-resolution PNG image (Scale=2 makes it ultra crisp)
#fig.write_image('/content/drive/MyDrive/interactive_triple_line_chart.png', scale=2)

#fig.write_html("interactive_triple_line_chart.html")
fig.show()


In [14]:
import pandas as pd
import numpy as np
import plotly.express as px


path_cpi = '/content/drive/MyDrive/ti-corruption-perception-index.csv'
path_gni = '/content/drive/MyDrive/API_NY.GNP.PCAP.CD_DS2_en_csv_v2_230.csv'
path_pop = '/content/drive/MyDrive/POP_DPND.csv'

# load to df59
df_cpi_59 = pd.read_csv(path_cpi)
df_gni_59 = pd.read_csv(path_gni, skiprows=4)
df_pop_59 = pd.read_csv(path_pop, skiprows=4)

target_year_wb = '2024'
target_year_cpi = 2024

# clean and prepare Data
cpi_clean_59 = df_cpi_59[df_cpi_59['Year'] == target_year_cpi][['Entity', 'Code', 'Corruption Perceptions Index']].copy()
cpi_clean_59.rename(columns={'Corruption Perceptions Index': 'CPI', 'Entity': 'Country Name'}, inplace=True)

pop_clean_59 = df_pop_59[['Country Code', target_year_wb]].copy()
pop_clean_59.rename(columns={target_year_wb: 'Dependency_Ratio'}, inplace=True)

gni_clean_59 = df_gni_59[['Country Code', target_year_wb]].copy()
gni_clean_59.rename(columns={target_year_wb: 'GNI_Per_Capita'}, inplace=True)

df_merged_59 = cpi_clean_59.merge(pop_clean_59, left_on='Code', right_on='Country Code', how='inner')
df_merged_59 = df_merged_59.merge(gni_clean_59, on='Country Code', how='inner')
df_merged_59.dropna(subset=['CPI', 'Dependency_Ratio', 'GNI_Per_Capita'], inplace=True)

# clean income logic
conditions_income = [
    df_merged_59['GNI_Per_Capita'] >= 14005,
    (df_merged_59['GNI_Per_Capita'] >= 1146) & (df_merged_59['GNI_Per_Capita'] < 14005),
    df_merged_59['GNI_Per_Capita'] < 1146
]
choices_income = ['High Income', 'Mid Income', 'Low Income']
df_merged_59['Income_Level'] = np.select(conditions_income, choices_income, default='Other')

# update group
conditions_category = [
    df_merged_59['Dependency_Ratio'] < 15,
    (df_merged_59['Dependency_Ratio'] >= 15) & (df_merged_59['Dependency_Ratio'] < 20),

    # Clean (CPI >= 50)
    (df_merged_59['Dependency_Ratio'] >= 30) & (df_merged_59['CPI'] >= 50) & (df_merged_59['Income_Level'] == 'High Income'),
    (df_merged_59['Dependency_Ratio'] >= 20) & (df_merged_59['Dependency_Ratio'] < 30) & (df_merged_59['CPI'] >= 50) & (df_merged_59['Income_Level'] == 'High Income'),
    (df_merged_59['Dependency_Ratio'] >= 20) & (df_merged_59['CPI'] >= 50) & (df_merged_59['Income_Level'] == 'Mid Income'),
    (df_merged_59['Dependency_Ratio'] >= 20) & (df_merged_59['CPI'] >= 50) & (df_merged_59['Income_Level'] == 'Low Income'),

    # Corrupt (CPI < 50)
    (df_merged_59['Dependency_Ratio'] >= 20) & (df_merged_59['CPI'] < 50) & (df_merged_59['Income_Level'] == 'High Income'),
    (df_merged_59['Dependency_Ratio'] >= 20) & (df_merged_59['CPI'] < 50) & (df_merged_59['Income_Level'] == 'Mid Income'),
    (df_merged_59['Dependency_Ratio'] >= 20) & (df_merged_59['CPI'] < 50) & (df_merged_59['Income_Level'] == 'Low Income')
]

choices_category = [
    "The Young World<br>(< 15% OADR)",
    "Approaching Aging<br>(15-20% OADR)",
    "Severe Aging<br>Clean, High Income",
    "Aging<br>Clean, High Income",
    "Aging<br>Clean, Mid Income",
    "Aging<br>Clean, Low Income",
    "Aging<br>Corrupt, High Income",
    "Aging<br>Corrupt, Mid Income",
    "Aging<br>Corrupt, Low Income"
]

df_merged_59['Category'] = np.select(conditions_category, choices_category, default='Uncategorized')

# force the order
master_order = [
    "Severe Aging<br>Clean, High Income", "Aging<br>Clean, High Income",
    "Aging<br>Clean, Mid Income", "Aging<br>Clean, Low Income",
    "Aging<br>Corrupt, High Income", "Aging<br>Corrupt, Mid Income", "Aging<br>Corrupt, Low Income",
    "Approaching Aging<br>(15-20% OADR)", "The Young World<br>(< 15% OADR)"
]

df_merged_59 = df_merged_59[df_merged_59['Category'] != 'Uncategorized']
actual_categories = df_merged_59['Category'].unique()
category_order = [cat for cat in master_order if cat in actual_categories]

# px.box
fig_59 = px.box(
    df_merged_59,
    x='Category',
    y='Dependency_Ratio',
    color='Category',
    points='all',
    hover_data=['Country Name', 'CPI', 'GNI_Per_Capita', 'Income_Level'],
    category_orders={"Category": category_order},
    color_discrete_map={
        "Severe Aging<br>Clean, High Income": "#1E3A8A",
        "Aging<br>Clean, High Income": "#38BDF8",
        "Aging<br>Clean, Mid Income": "#059669",
        "Aging<br>Clean, Low Income": "#A7F3D0",
        "Aging<br>Corrupt, High Income": "#B45309",
        "Aging<br>Corrupt, Mid Income": "#EA580C",
        "Aging<br>Corrupt, Low Income": "#DC2626",
        "Approaching Aging<br>(15-20% OADR)": "#8B5CF6",
        "The Young World<br>(< 15% OADR)": "#9CA3AF"
    }
)

fig_59.update_traces(width=0.45, marker=dict(size=6, opacity=0.85))

# visual formatting
fig_59.update_layout(
    plot_bgcolor='#FAFAFA',
    paper_bgcolor='white',
    yaxis=dict(
        title=dict(text='Old Age Dependency Ratio (OADR, %)', font=dict(color='black', size=18, family='Arial')),
        gridcolor='lightgray',
        gridwidth=1,
        griddash='dot',
        tickfont=dict(color='gray', size=16, family='Arial')
    ),
    xaxis=dict(
        title=None,
        tickangle=0,
        tickfont=dict(color='black', size=16, family='Arial')
    ),
    showlegend=False,
    width=1500,
    height=900,
    title=dict(
        text="Demographic Destiny: Wealth, Governance, and the Global Aging Curve<br><sup>High-income, transparent nations lead the aging trend, while corrupt and developing economies face a looming demographic trap.</sup>",
        font=dict(size=28, color='black', family='Arial'),
        x=0.05
    ),
    # work on bottom margin slightly (b=140) to accommodate the pushed-down text
    margin=dict(b=140, t=160, l=80, r=40)
)


fig_59.add_annotation(
    text="<b>Source:</b> World Bank & Transparency International, 2024. | <b>Note:</b> 'Clean' = CPI Score ≥ 50; 'Corrupt' = CPI < 50.",
    xref="paper", yref="paper",
    x=0.5, y=-0.16, # pushed further down from -0.12 to -0.16
    showarrow=False,
    font=dict(size=15, color="gray"),
    xanchor="center", yanchor="top",
    align="center"
)

fig_59.show()


fig_59.write_html('/content/drive/MyDrive/BoxPlot_3indicators_Demographic_Final.html')